# Resultados

Comenzamos cargando las librerías necesarias y cargando el rendimiendo de los modelos de aprendizaje.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Cargar resultados
results_path = 'resultados_eval/evaluacion_resultados.csv'
if not os.path.exists(results_path):
    print(f"Error: No se encuentra el archivo {results_path}. Ejecuta primero eval.ipynb.")
else:
    df = pd.read_csv(results_path)
    print("Datos cargados correctamente.")
    display(df.head())

## Generación de Tablas Resumen (Media $\pm$ Desviación Típica)

Generamos unas tablas con un resumen de los datos obtenidos en la evaluación de los modelos que contiene la media y la desviación de cada método en cada dataset para cada estadística.

In [ ]:
# Definimos las métricas
metrics = ['Fm', 'Acc', 'S', 'SP', 'RC', 'PR', 'FNR', 'FPR', 'AUC']

# Calculamos las medias y desviaciones
grouped = df.groupby(['Dataset', 'Model'])[metrics]
means = grouped.mean()
stds = grouped.std()

# Combinamos los dataframes anteriores
summary_df = means.round(4).astype(str) + " ± " + stds.round(4).astype(str)

# Para que se vea mejor, reseteamos el índice
summary_df.reset_index(inplace=True)

display(summary_df)

### Tablas por Método

Mostramos una tabla para cada uno de los métodos usados que contiene todas las métricas para cada dataset.

In [ ]:
# Cogemos todos los métodos únicos
methods = summary_df['Model'].unique()

# Mostramos una tabla para cada método
for method in methods:
    print(f"\n--- Tabla Resumen para: {method} ---")
    method_df = summary_df[summary_df['Model'] == method].set_index('Dataset')
    display(method_df[metrics])

### Tabla Resumen F1-Score 

Creamos una tabla que tiene el F1-score de todos los métodos para cada dataset.

In [ ]:
# Tabla resumen F1-Score (Media ± Std) para todos los métodos
f1 = summary_df.pivot(index='Model', columns='Dataset', values='Fm')
print("\n--- Resumen F1-Score (Media ± Std) ---")
display(f1)

## Gráficas

Graficamos las gráficas pedidas en el enunciado de la práctica.

In [ ]:
def plot_scatter_metrics(x_col, y_col, xlabel, ylabel, title):
    plt.figure(figsize=(12, 8))
    
    # Usamos los valores medios para el plot
    plot_data = means.reset_index()
    
    sns.scatterplot(data=plot_data, x=x_col, y=y_col, hue='Model', style='Dataset', s=100)
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# 1. FN contra FP 
if 'FPR' in means.columns and 'FNR' in means.columns:
    plot_scatter_metrics('FPR', 'FNR', 'False Positive Rate (FPR)', 'False Negative Rate (FNR)', 'FNR vs FPR')

# 2. PR contra RC
if 'RC' in means.columns and 'PR' in means.columns:
    plot_scatter_metrics('RC', 'PR', 'Recall (RC)', 'Precision (PR)', 'Precision vs Recall')

# 3. ACC contra Fm
if 'Fm' in means.columns and 'Acc' in means.columns:
    plot_scatter_metrics('Fm', 'Acc', 'F1-Score (Fm)', 'Accuracy (Acc)', 'Accuracy vs F1-Score')